# 04 — Première modélisation supervisée ExpenseAI

## Objectif et limites

Ce notebook entraîne et compare une première sélection d'algorithmes de classification supervisée. L'unité de prédiction est une **ligne de dépense** et la classe positive est toujours :

- `0` : **Approuvée** ;
- `1` : **Refusée**.

La classe refusée est très rare. L'accuracy ne peut donc jamais être interprétée seule. Cette étape utilise des pipelines scikit-learn et une séparation respectant les groupes de notes de frais.

Cette première expérience n'effectue volontairement **ni GridSearchCV, ni RandomizedSearchCV, ni SMOTE, ni sous-échantillonnage, ni SHAP, ni sauvegarde de modèle, ni prédiction en base ou dans Streamlit**.

## 1. Imports et configuration

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display
from plotly.subplots import make_subplots
from sqlalchemy import text

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
pio.templates.default = "plotly_white"

RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from database.connection import create_db_engine

## 2. Chargement depuis PostgreSQL

PostgreSQL est l'unique source de cette modélisation. La vue `v_ml_expenses` reconstruit les variables préparées à partir des tables normalisées. La requête est en lecture seule et aucune donnée PostgreSQL n'est modifiée.

In [2]:
ML_QUERY = text("SELECT * FROM v_ml_expenses")

engine = create_db_engine()
try:
    with engine.connect() as connection:
        df = pd.read_sql(ML_QUERY, connection)
finally:
    engine.dispose()

# Conversion explicite des types utiles à scikit-learn.
numeric_columns_from_sql = [
    "amount_ttc",
    "tax_rate",
    "annee",
    "mois",
    "jour",
    "jour_semaine",
    "trimestre",
    "est_weekend",
    "target",
]
for column in numeric_columns_from_sql:
    df[column] = pd.to_numeric(df[column], errors="coerce")
df["billable"] = df["billable"].astype(int)

print("Données chargées depuis la vue PostgreSQL v_ml_expenses.")

Données chargées depuis la vue PostgreSQL v_ml_expenses.


## 3. Contrôles avant modélisation

In [3]:
target_counts = df["target"].value_counts().sort_index()
controls = pd.Series(
    {
        "Nombre de lignes": len(df),
        "Nombre de colonnes": len(df.columns),
        "Classes distinctes": sorted(df["target"].dropna().unique().tolist()),
        "Classe 0 — Approuvée": int(target_counts.get(0, 0)),
        "Classe 1 — Refusée": int(target_counts.get(1, 0)),
        "Valeurs manquantes totales": int(df.isna().sum().sum()),
        "Groupes de notes de frais": int(df["expense_group"].nunique()),
        "Types de dépense": int(df["type"].nunique()),
        "Modalités project_code": int(df["project_code"].nunique()),
    },
    name="Valeur",
)
display(controls.to_frame())

assert len(df) == 7070, "Le volume attendu est de 7 070 lignes."
assert set(df["target"].unique()) == {0, 1}, "La cible doit contenir uniquement 0 et 1."
assert target_counts.to_dict() == {0: 6956, 1: 114}, "La répartition de la cible est inattendue."
assert df.isna().sum().sum() == 0, "Des valeurs manquantes subsistent dans la vue ML."

,Valeur
Nombre de lignes,7070
Nombre de colonnes,13
Classes distinctes,"[0, 1]"
Classe 0 — Approuvée,6956
Classe 1 — Refusée,114
Valeurs manquantes totales,0
Groupes de notes de frais,6253
Types de dépense,33
Modalités project_code,195


## 4. Définition de `X`, `y` et `groups`

Les variables initiales sont : `type`, `amount_ttc`, `billable`, `project_code`, `tax_rate`, `annee`, `mois`, `jour`, `jour_semaine`, `trimestre` et `est_weekend`.

- `expense_group` est conservé séparément dans `groups` : il identifie une note de frais pouvant contenir plusieurs lignes. L'utiliser comme feature permettrait au modèle de mémoriser des identifiants et favoriserait une fuite entre observations liées.
- `Date d'approbation` et `Motif du refus` ne figurent ni dans la vue ni dans les features : ces informations sont potentiellement connues seulement après la décision.
- Aucune information postérieure à la validation n'est présente dans `X`.

In [4]:
FEATURES = [
    "type",
    "amount_ttc",
    "billable",
    "project_code",
    "tax_rate",
    "annee",
    "mois",
    "jour",
    "jour_semaine",
    "trimestre",
    "est_weekend",
]

X = df[FEATURES].copy()
y = df["target"].astype(int).copy()
groups = df["expense_group"].astype(str).copy()

assert "expense_group" not in X.columns
assert "target" not in X.columns
display(pd.DataFrame({"Feature": X.columns, "Type pandas": X.dtypes.astype(str)}))

,Feature,Type pandas
type,type,object
amount_ttc,amount_ttc,float64
billable,billable,int64
project_code,project_code,object
tax_rate,tax_rate,float64
annee,annee,int64
mois,mois,int64
jour,jour,int64
jour_semaine,jour_semaine,int64
trimestre,trimestre,int64


## 5. Création du jeu de test indépendant

Un simple `train_test_split` ligne par ligne serait incorrect : plusieurs lignes d'une même note pourraient se retrouver des deux côtés. Le premier fold d'un `StratifiedGroupKFold` à cinq folds est donc figé comme test final. Le reste constitue le train.

Le choix du modèle et l'expérience sur `project_code` seront réalisés **uniquement par validation croisée sur le train**. Le test reste isolé jusqu'à l'évaluation finale.

In [5]:
outer_split = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)
train_indices, test_indices = next(outer_split.split(X, y, groups=groups))

X_train = X.iloc[train_indices].copy()
X_test = X.iloc[test_indices].copy()
y_train = y.iloc[train_indices].copy()
y_test = y.iloc[test_indices].copy()
groups_train = groups.iloc[train_indices].copy()
groups_test = groups.iloc[test_indices].copy()

groups_are_disjoint = set(groups_train).isdisjoint(set(groups_test))
split_report = pd.DataFrame(
    {
        "Jeu": ["Train", "Test"],
        "Lignes": [len(X_train), len(X_test)],
        "Groupes": [groups_train.nunique(), groups_test.nunique()],
        "Refus": [int(y_train.sum()), int(y_test.sum())],
        "Taux de refus (%)": [y_train.mean() * 100, y_test.mean() * 100],
    }
)
display(split_report)
print("Groupes train/test disjoints :", groups_are_disjoint)

assert groups_are_disjoint, "Un groupe est présent à la fois dans le train et le test."
assert len(X_train) + len(X_test) == len(X)
assert y_test.sum() > 0, "Le jeu de test ne contient aucun refus."

,Jeu,Lignes,Groupes,Refus,Taux de refus (%)
0,Train,5656,5001,91,1.6089
1,Test,1414,1252,23,1.6266


Groupes train/test disjoints : True


> **Isolement du test.** À partir de ce point, `X_test` et `y_test` ne servent ni à apprendre le preprocessing, ni à sélectionner un algorithme, ni à comparer la présence de `project_code`. Ils ne seront ouverts qu'après la validation croisée du train.

## 6. Preprocessing scikit-learn

In [6]:
CATEGORICAL_FEATURES = ["type", "project_code"]
NUMERIC_FEATURES = [
    "amount_ttc",
    "tax_rate",
    "annee",
    "mois",
    "jour",
    "jour_semaine",
    "trimestre",
    "est_weekend",
    "billable",
]


def build_preprocessor(include_project_code: bool, scale_numeric: bool) -> ColumnTransformer:
    """Construit le preprocessing sans l'apprendre avant le split."""
    categorical_features = ["type"]
    if include_project_code:
        categorical_features.append("project_code")

    numeric_transformer = (
        Pipeline([("standardisation", StandardScaler())])
        if scale_numeric
        else "passthrough"
    )
    return ColumnTransformer(
        transformers=[
            (
                "categories",
                OneHotEncoder(handle_unknown="ignore"),
                categorical_features,
            ),
            ("numeriques", numeric_transformer, NUMERIC_FEATURES),
        ],
        remainder="drop",
    )


display(
    pd.DataFrame(
        {
            "Famille": ["Catégorielles", "Numériques"],
            "Variables": [", ".join(CATEGORICAL_FEATURES), ", ".join(NUMERIC_FEATURES)],
            "Traitement": [
                "OneHotEncoder(handle_unknown='ignore')",
                "StandardScaler pour la régression logistique ; passthrough pour les arbres",
            ],
        }
    )
)

,Famille,Variables,Traitement
0,Catégorielles,"type, project_code",OneHotEncoder(handle_unknown='ignore')
1,Numériques,"amount_ttc, tax_rate, annee, mois, jour, jour_...",StandardScaler pour la régression logistique ;...


Le `ColumnTransformer` est placé à l'intérieur de chaque `Pipeline`. Il sera donc appris séparément dans chaque fold d'entraînement. Aucun `pd.get_dummies()` n'est appliqué avant le split.

## 7. Modèles et premières configurations

In [7]:
MODEL_NAMES = [
    "DummyClassifier",
    "Régression logistique",
    "Arbre de décision",
    "Forêt aléatoire",
]


def build_estimator(model_name: str):
    """Retourne la configuration initiale demandée, sans optimisation."""
    if model_name == "DummyClassifier":
        return DummyClassifier(strategy="most_frequent")
    if model_name == "Régression logistique":
        return LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )
    if model_name == "Arbre de décision":
        return DecisionTreeClassifier(
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )
    if model_name == "Forêt aléatoire":
        return RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    raise ValueError(f"Modèle inconnu : {model_name}")


def build_model_pipeline(model_name: str, include_project_code: bool = True) -> Pipeline:
    """Assemble preprocessing et estimateur dans une chaîne sans fuite."""
    scale_numeric = model_name == "Régression logistique"
    return Pipeline(
        steps=[
            (
                "preprocessing",
                build_preprocessor(
                    include_project_code=include_project_code,
                    scale_numeric=scale_numeric,
                ),
            ),
            ("modele", build_estimator(model_name)),
        ]
    )


model_parameters = pd.DataFrame(
    {
        "Modèle": MODEL_NAMES,
        "Rôle / configuration": [
            "Baseline prédisant toujours la classe majoritaire",
            "max_iter=2000, class_weight='balanced'",
            "class_weight='balanced'",
            "300 arbres, class_weight='balanced_subsample'",
        ],
    }
)
display(model_parameters)

,Modèle,Rôle / configuration
0,DummyClassifier,Baseline prédisant toujours la classe majoritaire
1,Régression logistique,"max_iter=2000, class_weight='balanced'"
2,Arbre de décision,class_weight='balanced'
3,Forêt aléatoire,"300 arbres, class_weight='balanced_subsample'"


## 8. Validation croisée groupée sur le train

La validation croisée suivante utilise exclusivement `X_train`, `y_train` et `groups_train`. La classe positive des scores de précision, rappel et F1 est `Refusée = 1`. Les scores d'entraînement sont également calculés pour repérer d'éventuels écarts importants avec la validation.

In [8]:
inner_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

scoring = {
    "recall": make_scorer(recall_score, pos_label=1, zero_division=0),
    "precision": make_scorer(precision_score, pos_label=1, zero_division=0),
    "f1": make_scorer(f1_score, pos_label=1, zero_division=0),
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "balanced_accuracy": "balanced_accuracy",
}

cv_raw_results = {}
cv_rows = []
for model_name in MODEL_NAMES:
    pipeline = build_model_pipeline(model_name)
    result = cross_validate(
        pipeline,
        X_train,
        y_train,
        groups=groups_train,
        cv=inner_cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=1,
        error_score="raise",
    )
    cv_raw_results[model_name] = result
    row = {"Modèle": model_name}
    for metric_name in scoring:
        row[f"{metric_name}_moyenne"] = result[f"test_{metric_name}"].mean()
        row[f"{metric_name}_ecart_type"] = result[f"test_{metric_name}"].std()
        row[f"{metric_name}_train_moyenne"] = result[f"train_{metric_name}"].mean()
    cv_rows.append(row)

cv_summary = pd.DataFrame(cv_rows)
display(cv_summary)

,Modèle,recall_moyenne,recall_ecart_type,recall_train_moyenne,precision_moyenne,precision_ecart_type,precision_train_moyenne,f1_moyenne,f1_ecart_type,f1_train_moyenne,roc_auc_moyenne,roc_auc_ecart_type,roc_auc_train_moyenne,average_precision_moyenne,average_precision_ecart_type,average_precision_train_moyenne,balanced_accuracy_moyenne,balanced_accuracy_ecart_type,balanced_accuracy_train_moyenne
0,DummyClassifier,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.5000,0.0000,0.5000,0.0161,0.0003,0.0161,0.5000,0.0000,0.5000
1,Régression logistique,0.6246,0.1218,0.8763,0.0528,0.0078,0.0732,0.0973,0.0146,0.1351,0.7795,0.0706,0.9213,0.1018,0.0404,0.1368,0.7208,0.0544,0.8468
2,Arbre de décision,0.1649,0.0610,1.0000,0.1952,0.1011,0.9892,0.1757,0.0759,0.9945,0.5757,0.0327,1.0000,0.0510,0.0236,0.9999,0.5757,0.0327,0.9999
3,Forêt aléatoire,0.0444,0.0416,1.0000,0.5000,0.4472,0.9892,0.0811,0.0751,0.9945,0.7762,0.0407,1.0000,0.1497,0.0370,0.9964,0.5221,0.0208,0.9999


In [9]:
cv_display = cv_summary[
    [
        "Modèle",
        "balanced_accuracy_moyenne",
        "balanced_accuracy_ecart_type",
        "precision_moyenne",
        "precision_ecart_type",
        "recall_moyenne",
        "recall_ecart_type",
        "f1_moyenne",
        "f1_ecart_type",
        "roc_auc_moyenne",
        "roc_auc_ecart_type",
        "average_precision_moyenne",
        "average_precision_ecart_type",
    ]
].copy()
display(cv_display.sort_values("average_precision_moyenne", ascending=False))

# La PR-AUC est la métrique principale de présélection car la classe positive est rare.
true_model_cv = cv_summary[cv_summary["Modèle"] != "DummyClassifier"].copy()
true_model_cv = true_model_cv.sort_values(
    ["average_precision_moyenne", "recall_moyenne"],
    ascending=False,
)
promising_model_name = true_model_cv.iloc[0]["Modèle"]
print("Modèle le plus prometteur selon la PR-AUC moyenne sur le train :", promising_model_name)

,Modèle,balanced_accuracy_moyenne,balanced_accuracy_ecart_type,precision_moyenne,precision_ecart_type,recall_moyenne,recall_ecart_type,f1_moyenne,f1_ecart_type,roc_auc_moyenne,roc_auc_ecart_type,average_precision_moyenne,average_precision_ecart_type
3,Forêt aléatoire,0.5221,0.0208,0.5000,0.4472,0.0444,0.0416,0.0811,0.0751,0.7762,0.0407,0.1497,0.0370
1,Régression logistique,0.7208,0.0544,0.0528,0.0078,0.6246,0.1218,0.0973,0.0146,0.7795,0.0706,0.1018,0.0404
2,Arbre de décision,0.5757,0.0327,0.1952,0.1011,0.1649,0.0610,0.1757,0.0759,0.5757,0.0327,0.0510,0.0236
0,DummyClassifier,0.5000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.5000,0.0000,0.0161,0.0003


Modèle le plus prometteur selon la PR-AUC moyenne sur le train : Forêt aléatoire


Le modèle présélectionné ci-dessus résulte uniquement de la validation croisée groupée du train. Il ne s'agit pas encore d'un modèle final et aucune optimisation d'hyperparamètres n'a été réalisée.

## 9. Expérience complémentaire sur `project_code`

In [10]:
project_experiment_rows = []
project_experiment_raw = {}
for include_project_code, label in [
    (True, "Avec project_code"),
    (False, "Sans project_code"),
]:
    experiment_features = FEATURES if include_project_code else [
        feature for feature in FEATURES if feature != "project_code"
    ]
    result = cross_validate(
        build_model_pipeline(
            promising_model_name,
            include_project_code=include_project_code,
        ),
        X_train[experiment_features],
        y_train,
        groups=groups_train,
        cv=inner_cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=1,
        error_score="raise",
    )
    project_experiment_raw[label] = result
    project_experiment_rows.append(
        {
            "Configuration": label,
            "Balanced Accuracy moyenne": result["test_balanced_accuracy"].mean(),
            "Precision moyenne": result["test_precision"].mean(),
            "Recall moyen": result["test_recall"].mean(),
            "F1 moyen": result["test_f1"].mean(),
            "ROC-AUC moyenne": result["test_roc_auc"].mean(),
            "PR-AUC moyenne": result["test_average_precision"].mean(),
            "PR-AUC écart-type": result["test_average_precision"].std(),
            "PR-AUC train moyenne": result["train_average_precision"].mean(),
        }
    )

project_experiment = pd.DataFrame(project_experiment_rows)
display(project_experiment)

project_pr_auc_with = project_experiment.loc[
    project_experiment["Configuration"].eq("Avec project_code"), "PR-AUC moyenne"
].iloc[0]
project_pr_auc_without = project_experiment.loc[
    project_experiment["Configuration"].eq("Sans project_code"), "PR-AUC moyenne"
].iloc[0]
project_pr_auc_delta = project_pr_auc_with - project_pr_auc_without

display(
    Markdown(
        f"Pour **{promising_model_name}**, l'écart moyen de PR-AUC "
        f"(avec − sans `project_code`) vaut **{project_pr_auc_delta:+.4f}**. "
        "Cette expérience groupée mesure un apport éventuel, mais ne suffit pas à "
        "prendre une décision définitive sur cette variable à forte cardinalité."
    )
)

,Configuration,Balanced Accuracy moyenne,Precision moyenne,Recall moyen,F1 moyen,ROC-AUC moyenne,PR-AUC moyenne,PR-AUC écart-type,PR-AUC train moyenne
0,Avec project_code,0.5221,0.5000,0.0444,0.0811,0.7762,0.1497,0.0370,0.9964
1,Sans project_code,0.5108,0.3000,0.0222,0.0411,0.7837,0.1313,0.0632,0.9939


Pour **Forêt aléatoire**, l'écart moyen de PR-AUC (avec − sans `project_code`) vaut **+0.0185**. Cette expérience groupée mesure un apport éventuel, mais ne suffit pas à prendre une décision définitive sur cette variable à forte cardinalité.

## 10. Ouverture unique du jeu de test final

Les comparaisons de validation et l'expérience `project_code` sont maintenant terminées. Chaque pipeline est appris une fois sur tout le train, puis évalué une seule fois sur le test groupé indépendant. Le seuil de décision reste fixé à **0,5**.

In [11]:
fitted_models = {}
test_probabilities = {}
test_predictions = {}
classification_reports = {}
test_rows = []

for model_name in MODEL_NAMES:
    pipeline = build_model_pipeline(model_name)
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]

    fitted_models[model_name] = pipeline
    test_predictions[model_name] = predictions
    test_probabilities[model_name] = probabilities
    classification_reports[model_name] = pd.DataFrame(
        classification_report(
            y_test,
            predictions,
            labels=[0, 1],
            target_names=["Approuvée", "Refusée"],
            output_dict=True,
            zero_division=0,
        )
    ).transpose()

    test_rows.append(
        {
            "Modèle": model_name,
            "Accuracy": accuracy_score(y_test, predictions),
            "Balanced Accuracy": balanced_accuracy_score(y_test, predictions),
            "Precision — Refusée": precision_score(
                y_test, predictions, pos_label=1, zero_division=0
            ),
            "Recall — Refusée": recall_score(
                y_test, predictions, pos_label=1, zero_division=0
            ),
            "F1 — Refusée": f1_score(
                y_test, predictions, pos_label=1, zero_division=0
            ),
            "ROC-AUC": roc_auc_score(y_test, probabilities),
            "PR-AUC": average_precision_score(y_test, probabilities),
        }
    )

test_comparison = pd.DataFrame(test_rows)
display(test_comparison)

,Modèle,Accuracy,Balanced Accuracy,Precision — Refusée,Recall — Refusée,F1 — Refusée,ROC-AUC,PR-AUC
0,DummyClassifier,0.9837,0.5000,0.0000,0.0000,0.0000,0.5000,0.0163
1,Régression logistique,0.7977,0.8117,0.0631,0.8261,0.1173,0.8535,0.0858
2,Arbre de décision,0.9724,0.6225,0.2143,0.2609,0.2353,0.6225,0.0679
3,Forêt aléatoire,0.9844,0.5431,0.6667,0.0870,0.1538,0.8517,0.2344


## 11. Tableau final de comparaison

In [12]:
dummy_test = test_comparison[test_comparison["Modèle"].eq("DummyClassifier")]
real_test_comparison = test_comparison[
    ~test_comparison["Modèle"].eq("DummyClassifier")
].sort_values(["PR-AUC", "Recall — Refusée"], ascending=False)

final_comparison = pd.concat([dummy_test, real_test_comparison], ignore_index=True)
display(final_comparison.style.format({column: "{:.4f}" for column in final_comparison.columns if column != "Modèle"}))

print(
    "Les vrais modèles sont triés selon la PR-AUC, métrique adaptée à une classe "
    "positive très minoritaire. Ce tri descriptif sur le test ne remplace pas la "
    "présélection effectuée auparavant sur le train."
)

,Modèle,Accuracy,Balanced Accuracy,Precision — Refusée,Recall — Refusée,F1 — Refusée,ROC-AUC,PR-AUC
0,DummyClassifier,0.9837,0.5000,0.0000,0.0000,0.0000,0.5000,0.0163
1,Forêt aléatoire,0.9844,0.5431,0.6667,0.0870,0.1538,0.8517,0.2344
2,Régression logistique,0.7977,0.8117,0.0631,0.8261,0.1173,0.8535,0.0858
3,Arbre de décision,0.9724,0.6225,0.2143,0.2609,0.2353,0.6225,0.0679


Les vrais modèles sont triés selon la PR-AUC, métrique adaptée à une classe positive très minoritaire. Ce tri descriptif sur le test ne remplace pas la présélection effectuée auparavant sur le train.


## 12. Rapports de classification

In [13]:
for model_name in MODEL_NAMES:
    display(Markdown(f"### {model_name}"))
    display(classification_reports[model_name])

### DummyClassifier

,precision,recall,f1-score,support
Approuvée,0.9837,1.0000,0.9918,"1,391.0000"
Refusée,0.0000,0.0000,0.0000,23.0000
accuracy,0.9837,0.9837,0.9837,0.9837
macro avg,0.4919,0.5000,0.4959,"1,414.0000"
weighted avg,0.9677,0.9837,0.9757,"1,414.0000"


### Régression logistique

,precision,recall,f1-score,support
Approuvée,0.9964,0.7973,0.8858,"1,391.0000"
Refusée,0.0631,0.8261,0.1173,23.0000
accuracy,0.7977,0.7977,0.7977,0.7977
macro avg,0.5298,0.8117,0.5015,"1,414.0000"
weighted avg,0.9812,0.7977,0.8733,"1,414.0000"


### Arbre de décision

,precision,recall,f1-score,support
Approuvée,0.9877,0.9842,0.9860,"1,391.0000"
Refusée,0.2143,0.2609,0.2353,23.0000
accuracy,0.9724,0.9724,0.9724,0.9724
macro avg,0.6010,0.6225,0.6106,"1,414.0000"
weighted avg,0.9752,0.9724,0.9737,"1,414.0000"


### Forêt aléatoire

,precision,recall,f1-score,support
Approuvée,0.9851,0.9993,0.9921,"1,391.0000"
Refusée,0.6667,0.0870,0.1538,23.0000
accuracy,0.9844,0.9844,0.9844,0.9844
macro avg,0.8259,0.5431,0.5730,"1,414.0000"
weighted avg,0.9799,0.9844,0.9785,"1,414.0000"


## 13. Matrices de confusion

Pour la classe positive `Refusée = 1` :

- **True Negative (TN)** : dépense approuvée correctement prédite approuvée ;
- **False Positive (FP)** : dépense approuvée signalée à tort comme refusée ;
- **False Negative (FN)** : dépense réellement refusée mais prédite approuvée ;
- **True Positive (TP)** : dépense refusée correctement détectée.

Dans ExpenseAI, les **False Negatives** méritent une attention particulière : ils correspondent à des dépenses historiquement refusées que l'outil laisserait passer sans alerte au valideur humain.

In [14]:
confusion_figure = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=MODEL_NAMES,
    horizontal_spacing=0.14,
    vertical_spacing=0.18,
)

for position, model_name in enumerate(MODEL_NAMES):
    row = position // 2 + 1
    column = position % 2 + 1
    matrix = confusion_matrix(y_test, test_predictions[model_name], labels=[0, 1])
    confusion_figure.add_trace(
        go.Heatmap(
            z=matrix,
            x=["Prédite approuvée", "Prédite refusée"],
            y=["Réelle approuvée", "Réelle refusée"],
            text=matrix,
            texttemplate="%{text}",
            colorscale="Blues",
            showscale=False,
            hovertemplate="%{y}<br>%{x}<br>Nombre : %{z}<extra></extra>",
        ),
        row=row,
        col=column,
    )

confusion_figure.update_layout(
    title="Matrices de confusion sur le jeu de test groupé",
    height=760,
)
confusion_figure.show()

## 14. Courbes ROC et Precision–Recall

In [15]:
roc_figure = go.Figure()
for model_name in MODEL_NAMES:
    false_positive_rate, true_positive_rate, _ = roc_curve(
        y_test, test_probabilities[model_name]
    )
    auc_value = test_comparison.loc[
        test_comparison["Modèle"].eq(model_name), "ROC-AUC"
    ].iloc[0]
    roc_figure.add_trace(
        go.Scatter(
            x=false_positive_rate,
            y=true_positive_rate,
            mode="lines",
            name=f"{model_name} — AUC {auc_value:.3f}",
        )
    )
roc_figure.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        line={"dash": "dash", "color": "#64748B"},
        name="Hasard",
    )
)
roc_figure.update_layout(
    title="Courbes ROC sur le jeu de test",
    xaxis_title="Taux de faux positifs",
    yaxis_title="Taux de vrais positifs",
)
roc_figure.show()

In [16]:
pr_figure = go.Figure()
for model_name in MODEL_NAMES:
    precision_values, recall_values, _ = precision_recall_curve(
        y_test, test_probabilities[model_name]
    )
    pr_auc_value = test_comparison.loc[
        test_comparison["Modèle"].eq(model_name), "PR-AUC"
    ].iloc[0]
    pr_figure.add_trace(
        go.Scatter(
            x=recall_values,
            y=precision_values,
            mode="lines",
            name=f"{model_name} — PR-AUC {pr_auc_value:.3f}",
        )
    )
pr_figure.add_hline(
    y=y_test.mean(),
    line_dash="dash",
    line_color="#64748B",
    annotation_text="Prévalence des refus",
)
pr_figure.update_layout(
    title="Courbes Precision–Recall sur le jeu de test",
    xaxis_title="Recall — Refusée",
    yaxis_title="Precision — Refusée",
)
pr_figure.show()

La courbe ROC décrit la capacité de classement globale, mais elle peut paraître favorable lorsque les négatifs sont très nombreux. La courbe **Precision–Recall** est particulièrement informative ici, car elle se concentre sur la qualité de détection de la classe rare `Refusée`.

## 15. Pourquoi le DummyClassifier est indispensable

In [17]:
dummy_row = test_comparison[test_comparison["Modèle"].eq("DummyClassifier")].iloc[0]
display(
    Markdown(
        f"Le DummyClassifier atteint une accuracy de **{dummy_row['Accuracy']:.2%}**, "
        f"mais son recall pour la classe refusée est de **{dummy_row['Recall — Refusée']:.2%}**. "
        "En prédisant toujours « Approuvée », il paraît performant selon l'accuracy tout "
        "en détectant **zéro dépense refusée**. Cette baseline démontre pourquoi "
        "l'accuracy seule est trompeuse dans ExpenseAI."
    )
)

Le DummyClassifier atteint une accuracy de **98.37%**, mais son recall pour la classe refusée est de **0.00%**. En prédisant toujours « Approuvée », il paraît performant selon l'accuracy tout en détectant **zéro dépense refusée**. Cette baseline démontre pourquoi l'accuracy seule est trompeuse dans ExpenseAI.

## 16. Interprétation métier des performances

ExpenseAI est une solution d'**aide à la décision**. Une prédiction de refus sert principalement à attirer l'attention d'un valideur humain : elle ne remplace pas sa décision.

- Un **recall élevé** détecte davantage de dépenses susceptibles d'être refusées, mais peut augmenter les faux positifs et donc le volume de contrôles inutiles.
- Une **precision élevée** rend les alertes plus fiables, mais certaines dépenses réellement refusées peuvent ne pas être détectées.

Le seuil reste fixé à **0,5** pour cette première comparaison. Aucun seuil optimal n'est défini ici ; son choix futur devra refléter le coût métier relatif des faux positifs et des faux négatifs.

## 17. Analyse temporelle complémentaire — perspective méthodologique

Le dataset couvre environ une année. Un futur protocole pourra entraîner les modèles sur les périodes les plus anciennes et les tester sur les périodes les plus récentes. Ce test temporel permettrait d'évaluer un éventuel changement des types de dépenses, des projets ou des comportements de validation.

Ce protocole ne remplace pas le split groupé principal dans cette première version. Il devra lui aussi garantir qu'une même note de frais n'est pas répartie entre apprentissage et test.

## Synthèse de la première modélisation

In [18]:
def format_metric(value: float) -> str:
    """Formate une métrique avec une virgule décimale."""
    return f"{value:.4f}".replace(".", ",")


def format_count(value: int) -> str:
    """Formate un effectif avec une espace comme séparateur de milliers."""
    return f"{value:,}".replace(",", " ")


model_lines = []
for model_name in MODEL_NAMES[1:]:
    row = test_comparison[test_comparison["Modèle"].eq(model_name)].iloc[0]
    model_lines.append(
        f"- **{model_name}** — Balanced Accuracy {format_metric(row['Balanced Accuracy'])}, "
        f"precision refus {format_metric(row['Precision — Refusée'])}, "
        f"recall refus {format_metric(row['Recall — Refusée'])}, "
        f"F1 refus {format_metric(row['F1 — Refusée'])}, "
        f"ROC-AUC {format_metric(row['ROC-AUC'])}, PR-AUC {format_metric(row['PR-AUC'])}."
    )

overfit_messages = []
for _, row in true_model_cv.iterrows():
    gap = row["average_precision_train_moyenne"] - row["average_precision_moyenne"]
    if gap >= 0.10:
        overfit_messages.append(
            f"{row['Modèle']} présente un écart train–validation de PR-AUC de {gap:.3f}"
        )
overfit_text = (
    "; ".join(overfit_messages)
    if overfit_messages
    else "aucun écart train–validation de PR-AUC supérieur ou égal à 0,10 n'est observé"
)

if project_pr_auc_delta > 0.02:
    project_interpretation = (
        "project_code améliore la PR-AUC moyenne dans cette expérience, mais sa forte "
        "cardinalité et la variabilité des folds imposent une confirmation ultérieure"
    )
elif project_pr_auc_delta < -0.02:
    project_interpretation = (
        "project_code réduit la PR-AUC moyenne dans cette expérience et ne montre pas "
        "d'apport immédiat, sans justifier encore une exclusion définitive"
    )
else:
    project_interpretation = (
        "l'effet moyen de project_code sur la PR-AUC reste limité dans cette expérience ; "
        "aucune décision définitive ne doit être prise"
    )

cv_best = true_model_cv.iloc[0]
cv_recall_best = true_model_cv.sort_values("recall_moyenne", ascending=False).iloc[0]
summary = f"""
### Résultats réellement observés

- Le train contient **{format_count(len(X_train))} lignes et {int(y_train.sum())} refus**, contre **{format_count(len(X_test))} lignes et {int(y_test.sum())} refus** dans le test. Les groupes sont strictement disjoints.
- Le **DummyClassifier** atteint {dummy_row['Accuracy']:.2%} d'accuracy, mais une precision, un recall et un F1 nuls pour la classe refusée.
{chr(10).join(model_lines)}
- Selon la validation croisée groupée du train, **{promising_model_name}** obtient la meilleure PR-AUC moyenne ({format_metric(cv_best['average_precision_moyenne'])} ± {format_metric(cv_best['average_precision_ecart_type'])}), mais son recall moyen est de {format_metric(cv_best['recall_moyenne'])}.
- **{cv_recall_best['Modèle']}** obtient le meilleur recall moyen en validation ({format_metric(cv_recall_best['recall_moyenne'])}) et une Balanced Accuracy moyenne de {format_metric(cv_recall_best['balanced_accuracy_moyenne'])}. Ces deux modèles sont prometteurs pour des objectifs métier différents ; cette analyse n'utilise pas le test pour les sélectionner.
- Concernant le surapprentissage : {overfit_text}. Ces écarts restent des signaux exploratoires, pas un diagnostic définitif.
- Pour **{promising_model_name}**, l'écart de PR-AUC « avec − sans `project_code` » est de **{project_pr_auc_delta:+.4f}** : {project_interpretation}.

### Limites

- Seulement **{int(y_train.sum())} refus** sont disponibles dans le train et **{int(y_test.sum())}** dans le test : les métriques de la classe positive peuvent varier fortement avec quelques observations.
- Les configurations sont raisonnables mais non optimisées, et le seuil 0,5 n'est pas adapté à partir d'un coût métier formalisé.
- La période observée est courte et aucun test temporel n'est encore réalisé.
- Les résultats décrivent l'historique disponible ; ils ne démontrent aucune causalité et ne suffisent pas à automatiser une décision.

Aucun modèle ne doit être qualifié d'« excellent » sur la seule base de l'accuracy. ExpenseAI reste ici une aide destinée à prioriser la revue humaine.
"""
display(Markdown(summary))


### Résultats réellement observés

- Le train contient **5 656 lignes et 91 refus**, contre **1 414 lignes et 23 refus** dans le test. Les groupes sont strictement disjoints.
- Le **DummyClassifier** atteint 98.37% d'accuracy, mais une precision, un recall et un F1 nuls pour la classe refusée.
- **Régression logistique** — Balanced Accuracy 0,8117, precision refus 0,0631, recall refus 0,8261, F1 refus 0,1173, ROC-AUC 0,8535, PR-AUC 0,0858.
- **Arbre de décision** — Balanced Accuracy 0,6225, precision refus 0,2143, recall refus 0,2609, F1 refus 0,2353, ROC-AUC 0,6225, PR-AUC 0,0679.
- **Forêt aléatoire** — Balanced Accuracy 0,5431, precision refus 0,6667, recall refus 0,0870, F1 refus 0,1538, ROC-AUC 0,8517, PR-AUC 0,2344.
- Selon la validation croisée groupée du train, **Forêt aléatoire** obtient la meilleure PR-AUC moyenne (0,1497 ± 0,0370), mais son recall moyen est de 0,0444.
- **Régression logistique** obtient le meilleur recall moyen en validation (0,6246) et une Balanced Accuracy moyenne de 0,7208. Ces deux modèles sont prometteurs pour des objectifs métier différents ; cette analyse n'utilise pas le test pour les sélectionner.
- Concernant le surapprentissage : Forêt aléatoire présente un écart train–validation de PR-AUC de 0.847; Arbre de décision présente un écart train–validation de PR-AUC de 0.949. Ces écarts restent des signaux exploratoires, pas un diagnostic définitif.
- Pour **Forêt aléatoire**, l'écart de PR-AUC « avec − sans `project_code` » est de **+0.0185** : l'effet moyen de project_code sur la PR-AUC reste limité dans cette expérience ; aucune décision définitive ne doit être prise.

### Limites

- Seulement **91 refus** sont disponibles dans le train et **23** dans le test : les métriques de la classe positive peuvent varier fortement avec quelques observations.
- Les configurations sont raisonnables mais non optimisées, et le seuil 0,5 n'est pas adapté à partir d'un coût métier formalisé.
- La période observée est courte et aucun test temporel n'est encore réalisé.
- Les résultats décrivent l'historique disponible ; ils ne démontrent aucune causalité et ne suffisent pas à automatiser une décision.

Aucun modèle ne doit être qualifié d'« excellent » sur la seule base de l'accuracy. ExpenseAI reste ici une aide destinée à prioriser la revue humaine.


## 19. Préparation de l'étape suivante

Une étape ultérieure pourra étudier, dans un protocole toujours groupé et sans toucher au test final pendant la sélection :

- `GridSearchCV` ou `RandomizedSearchCV` ;
- le réglage des hyperparamètres ;
- la comparaison entre `class_weight`, sous-échantillonnage et SMOTE appliqués **uniquement au train** ;
- l'optimisation du seuil de décision selon les coûts métier ;
- SHAP pour l'explicabilité ;
- la sauvegarde contrôlée du pipeline final puis son intégration à Streamlit.

Aucune de ces opérations n'est réalisée dans ce notebook.